<a href="https://colab.research.google.com/github/kawastony/Quadratic-Mechanism-Lens/blob/main/Exploration_tifa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

"""
TIFA: Thawing Inflation-Flavored Axion
Complete self-contained framework
"""
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────
# CONSTANTS
# ─────────────────────────────────
MPL       = 1.0
RHO_CRIT  = 3.0
OMEGA_M   = 0.310
OMEGA_R   = 9.0e-5
OMEGA_DE  = 0.689
RHO_M0    = OMEGA_M * RHO_CRIT
RHO_R0    = OMEGA_R * RHO_CRIT
RHO_C_LQC = 100.0 * RHO_CRIT

FIDUCIAL = {'f': 0.5, 'Lambda4': 2.067, 'phi0_pi': 0.8}

DESI = {
    'w0': -0.827, 'w0_err': 0.060,
    'wa': -0.750, 'wa_err': 0.290,
}

# ─────────────────────────────────
# POTENTIAL
# ─────────────────────────────────
def V(phi, f, L):
    return L * (1.0 - np.cos(phi / f))

def dV(phi, f, L):
    return (L / f) * np.sin(phi / f)

def d2V(phi, f, L):
    return -(L / f**2) * np.cos(phi / f)

def grad_over_V(phi, f, L):
    v = V(phi, f, L)
    return abs(dV(phi, f, L)) / v if abs(v) > 1e-30 else 0.0

# ─────────────────────────────────
# BACKGROUND SOLVER
# ─────────────────────────────────
def solve_background(f, Lambda4, phi0_pi,
                     N_start=-11.0,
                     N_end=0.0,
                     n_points=1000):
    phi0  = phi0_pi * np.pi * f
    dphi0 = 0.0

    def rhs(N, y):
        phi, dphi = y
        rho_m   = RHO_M0 * np.exp(-3*N)
        rho_r   = RHO_R0 * np.exp(-4*N)
        rho_phi = 0.5*dphi**2 + V(phi, f, Lambda4)
        rho_tot = rho_m + rho_r + rho_phi
        lqc     = max(1.0 - rho_tot/RHO_C_LQC, 1e-30)
        H2      = max((rho_tot/3.0)*lqc, 1e-30)
        H       = np.sqrt(H2)
        drho_tot = -3*rho_m - 4*rho_r - 3*dphi**2
        dH = ((drho_tot/(6*H))*lqc
              - (rho_tot/3)*(drho_tot/RHO_C_LQC))
        dH /= max(2*H, 1e-30)
        eps   = dH/H if H > 1e-15 else 0.0
        d2phi = -(3+eps)*dphi - dV(phi, f, Lambda4)/H2
        return [dphi, d2phi]

    sol = solve_ivp(
        rhs, (N_start, N_end), [phi0, dphi0],
        t_eval=np.linspace(N_start, N_end, n_points),
        method='Radau', rtol=1e-8, atol=1e-10)

    if not sol.success:
        return None

    N_arr    = sol.t
    phi_arr  = sol.y[0]
    dphi_arr = sol.y[1]
    z_arr    = np.exp(-N_arr) - 1.0

    KE      = 0.5*dphi_arr**2
    PE      = V(phi_arr, f, Lambda4)
    rho_phi = KE + PE
    p_phi   = KE - PE
    w_arr   = np.where(rho_phi > 1e-30, p_phi/rho_phi, -1.0)

    rho_tot_arr  = RHO_M0*np.exp(-3*N_arr) + RHO_R0*np.exp(-4*N_arr) + rho_phi
    Omega_DE_arr = np.where(rho_tot_arr > 1e-30, rho_phi/rho_tot_arr, 0.0)

    w0 = float(w_arr[-1])
    wa = float(w_arr[0] - w_arr[-1])

    return {
        'N': N_arr, 'z': z_arr,
        'phi': phi_arr, 'dphi': dphi_arr,
        'w': w_arr, 'w0': w0, 'wa': wa,
        'Omega_DE': Omega_DE_arr,
        'rho_phi': rho_phi,
        'success': True,
    }

# ─────────────────────────────────
# RIDGE SCANNER
# ─────────────────────────────────
def scan_ridge(f_range=(0.3, 0.9), L_range=(0.8, 1.8),
               n_grid=20, phi0_pi=0.495,
               w0_band=(-0.95, -0.85)):
    f_vals = np.linspace(*f_range, n_grid)
    L_vals = np.linspace(*L_range, n_grid)
    ridge_f, ridge_L = [], []
    grid_w0 = np.full((n_grid, n_grid), np.nan)
    print(f"Scanning {n_grid}x{n_grid} grid...")
    for i, f in enumerate(f_vals):
        for j, L in enumerate(L_vals):
            sol = solve_background(f, L, phi0_pi)
            if sol is None:
                continue
            w0 = sol['w0']
            grid_w0[i, j] = w0
            if w0_band[0] <= w0 <= w0_band[1]:
                ridge_f.append(f)
                ridge_L.append(L)
    ridge_f = np.array(ridge_f)
    ridge_L = np.array(ridge_L)
    result  = {
        'f_vals': f_vals, 'L_vals': L_vals,
        'grid_w0': grid_w0,
        'ridge_f': ridge_f, 'ridge_L': ridge_L,
        'n_ridge': len(ridge_f),
    }
    if len(ridge_f) >= 4:
        def powerlaw(f, A, n):
            return A * f**n
        try:
            popt, pcov = curve_fit(
                powerlaw, ridge_f, ridge_L, p0=[0.82, 0.316])
            perr  = np.sqrt(np.diag(pcov))
            L_pred = powerlaw(ridge_f, *popt)
            ss_res = np.sum((ridge_L - L_pred)**2)
            ss_tot = np.sum((ridge_L - np.mean(ridge_L))**2)
            r2 = 1 - ss_res/ss_tot if ss_tot > 0 else 0.0
            result.update({
                'fit_A': float(popt[0]), 'fit_n': float(popt[1]),
                'fit_A_err': float(perr[0]), 'fit_n_err': float(perr[1]),
                'fit_R2': float(r2),
            })
            print(f"Ridge fit: Lambda4 = {popt[0]:.3f} * f^{popt[1]:.3f} (R2={r2:.4f})")
        except Exception as e:
            print(f"Fit failed: {e}")
    return result

# ─────────────────────────────────
# EFT DIAGNOSTICS
# ─────────────────────────────────
def eft_diagnostics(f, Lambda4, phi0_pi):
    phi0      = phi0_pi * np.pi * f
    delta_phi = abs(phi0)
    v_pp      = abs(d2V(phi0, f, Lambda4))
    coupling  = v_pp / Lambda4
    dV_over_V = (coupling**2 / (16*np.pi**2)) * np.log(max(Lambda4, 1e-30))
    v0        = V(phi0, f, Lambda4)
    cw_ratio  = dV_over_V / max(v0, 1e-30)
    eta_V     = d2V(phi0, f, Lambda4) / v0
    lqc_ratio = v0 / RHO_C_LQC
    return {
        'sub_Planck_f'        : f < MPL,
        'sub_Planck_excursion': delta_phi < MPL,
        'CW_ratio'            : cw_ratio,
        'eta_V'               : eta_V,
        'eta_V_negative'      : eta_V < 0,
        'lqc_ratio'           : lqc_ratio,
    }

# ─────────────────────────────────
# SWAMPLAND
# ─────────────────────────────────
def swampland_diagnostics(ridge_f, ridge_L,
                          phi0_pi=0.495,
                          c_values=(0.1, 0.3, 1.0)):
    results = {c: [] for c in c_values}
    slopes  = []
    for f, L in zip(ridge_f, ridge_L):
        phi0  = phi0_pi * np.pi * f
        slope = grad_over_V(phi0, f, L)
        slopes.append(slope)
        for c in c_values:
            results[c].append(slope >= c)
    return {
        'slopes'        : slopes,
        'slope_min'     : float(np.min(slopes)),
        'slope_max'     : float(np.max(slopes)),
        'slope_median'  : float(np.median(slopes)),
        'pass_fractions': {c: float(np.mean(results[c])) for c in c_values},
    }

# ─────────────────────────────────
# EARLY UNIVERSE SAFETY
# ─────────────────────────────────
def early_universe_safety(f, Lambda4, phi0_pi):
    sol = solve_background(f, Lambda4, phi0_pi,
                           N_start=-11.0, n_points=2000)
    if sol is None:
        return {'success': False}
    z_arr        = sol['z']
    Omega_DE_arr = sol['Omega_DE']
    idx_rec   = np.argmin(np.abs(z_arr - 1100))
    Omega_rec = float(Omega_DE_arr[idx_rec])
    z_BBN     = 3e8
    rho_r_BBN = OMEGA_R * (1 + z_BBN)**4
    phi0      = phi0_pi * np.pi * f
    Omega_BBN = V(phi0, f, Lambda4) / max(rho_r_BBN, 1e-30)
    return {
        'Omega_rec': Omega_rec,
        'Omega_BBN': Omega_BBN,
        'rec_safe' : Omega_rec < 1e-3,
        'BBN_safe' : Omega_BBN < 1e-2,
        'success'  : True,
    }

# ─────────────────────────────────
# LINEAR GROWTH
# ─────────────────────────────────
def linear_growth(f, Lambda4, phi0_pi, n_points=500):
    sol = solve_background(f, Lambda4, phi0_pi,
                           n_points=n_points)
    if sol is None:
        return {'success': False}
    N_arr   = sol['N']
    w_arr   = sol['w']
    rho_phi = sol['rho_phi']

    def growth_rhs(N, y):
        D, Dp = y
        idx    = np.argmin(np.abs(N_arr - N))
        rho_m  = RHO_M0 * np.exp(-3*N)
        rho_r  = RHO_R0 * np.exp(-4*N)
        rho_de = rho_phi[idx]
        rho_tot= rho_m + rho_r + rho_de
        H2     = max(rho_tot/3.0, 1e-30)
        p_de   = w_arr[idx] * rho_de
        q      = -0.5*(rho_m + 2*rho_r - 2*p_de) / max(3*H2, 1e-30)
        Om_m   = rho_m / (3*H2)
        return [Dp, -(2-q)*Dp + 1.5*Om_m*D]

    a_start  = np.exp(N_arr[0])
    y0       = [a_start, a_start]

    sol_tifa = solve_ivp(growth_rhs, (N_arr[0], N_arr[-1]),
                         y0, t_eval=N_arr,
                         method='Radau', rtol=1e-6, atol=1e-8)

    def growth_lcdm(N, y):
        D, Dp  = y
        rho_m  = OMEGA_M * np.exp(-3*N)
        rho_tot= rho_m + OMEGA_DE
        H2     = max(rho_tot/3.0, 1e-30)
        q      = -0.5*(-2*OMEGA_DE)/(3*H2)
        Om_m   = rho_m/(3*H2)
        return [Dp, -(2-q)*Dp + 1.5*Om_m*D]

    sol_lcdm = solve_ivp(growth_lcdm, (N_arr[0], N_arr[-1]),
                         y0, t_eval=N_arr,
                         method='Radau', rtol=1e-6, atol=1e-8)

    if not sol_tifa.success or not sol_lcdm.success:
        return {'success': False}

    D_tifa = sol_tifa.y[0] / sol_tifa.y[0][-1]
    D_lcdm = sol_lcdm.y[0] / sol_lcdm.y[0][-1]
    deviation = float(np.max(np.abs(D_tifa - D_lcdm) / np.abs(D_lcdm + 1e-30)))
    f_growth  = float(-np.gradient(np.log(D_tifa), N_arr)[-1])
    fsigma8   = f_growth * 0.811

    return {
        'D_tifa': D_tifa, 'D_lcdm': D_lcdm,
        'N_arr': N_arr,
        'deviation': deviation,
        'fsigma8': fsigma8,
        'success': True,
    }

# ─────────────────────────────────
# STATISTICAL COMPARISON
# ─────────────────────────────────
def statistical_comparison(w0, wa):
    chi2_lcdm = ((DESI['w0']+1.0)/DESI['w0_err'])**2 \
              + (DESI['wa']/DESI['wa_err'])**2
    chi2_tifa = ((DESI['w0']-w0)/DESI['w0_err'])**2 \
              + ((DESI['wa']-wa)/DESI['wa_err'])**2
    aic_lcdm  = chi2_lcdm + 2
    aic_tifa  = chi2_tifa  + 6
    return {
        'chi2_lcdm' : float(chi2_lcdm),
        'chi2_tifa' : float(chi2_tifa),
        'delta_chi2': float(chi2_tifa - chi2_lcdm),
        'aic_lcdm'  : float(aic_lcdm),
        'aic_tifa'  : float(aic_tifa),
        'delta_aic' : float(aic_tifa - aic_lcdm),
        'w0_tension': float(abs(w0-DESI['w0'])/DESI['w0_err']),
        'wa_tension': float(abs(wa-DESI['wa'])/DESI['wa_err']),
    }

# ─────────────────────────────────
# STRESS TEST
# ─────────────────────────────────
def stress_test(fiducial, n_samples=200, sigma=0.05):
    w0_vals, wa_vals = [], []
    rng = np.random.default_rng(42)
    for _ in range(n_samples):
        p = {k: v*(1+sigma*rng.standard_normal())
             for k, v in fiducial.items()}
        sol = solve_background(**p)
        if sol:
            w0_vals.append(sol['w0'])
            wa_vals.append(sol['wa'])
    w0_arr = np.array(w0_vals)
    if len(w0_arr) == 0:
        return {'n_converged': 0, 'w0_mean': np.nan,
                'w0_std': np.nan, 'desi_in_envelope': False}
    return {
        'n_converged'     : len(w0_arr),
        'w0_mean'         : float(np.mean(w0_arr)),
        'w0_std'          : float(np.std(w0_arr)),
        'w0_min'          : float(np.min(w0_arr)),
        'w0_max'          : float(np.max(w0_arr)),
        'wa_mean'         : float(np.mean(np.array(wa_vals))),
        'wa_std'          : float(np.std(np.array(wa_vals))),
        'desi_in_envelope': bool(np.min(w0_arr) <= DESI['w0'] <= np.max(w0_arr)),
    }

# ─────────────────────────────────
# AUDITOR
# ─────────────────────────────────
class TIFAConsistencyAuditor:
    def __init__(self, params=None):
        self.params  = params or FIDUCIAL
        self.results = {}

    def run_all(self, run_ridge=True, run_stress=True, n_grid=20):
        print("\n" + "="*50)
        print("TIFA CONSISTENCY AUDITOR")
        print("="*50)

        print("\n[1/6] Background evolution...")
        sol = solve_background(**self.params)
        if sol:
            self.results['background'] = {
                'w0': sol['w0'], 'wa': sol['wa'], 'success': True}
            print(f"  w0 = {sol['w0']:.4f}")
            print(f"  wa = {sol['wa']:.4f}")
        else:
            print("  FAILED")
            self.results['background'] = {'success': False}

        print("\n[2/6] EFT diagnostics...")
        eft = eft_diagnostics(**self.params)
        self.results['eft'] = eft
        print(f"  sub-Pl f:    {eft['sub_Planck_f']}")
        print(f"  sub-Pl Dphi: {eft['sub_Planck_excursion']}")
        print(f"  CW ratio:    {eft['CW_ratio']:.2e}")
        print(f"  eta_V < 0:   {eft['eta_V_negative']}")
        print(f"  LQC ratio:   {eft['lqc_ratio']:.2e}")

        print("\n[3/6] Early universe safety...")
        early = early_universe_safety(**self.params)
        self.results['early'] = early
        if early['success']:
            print(f"  Omega_DE(rec): {early['Omega_rec']:.2e}")
            print(f"  Omega_DE(BBN): {early['Omega_BBN']:.2e}")
            print(f"  rec safe:  {early['rec_safe']}")
            print(f"  BBN safe:  {early['BBN_safe']}")

        if run_ridge:
            print("\n[4/6] Ridge scan...")
            ridge = scan_ridge(
                phi0_pi=self.params['phi0_pi'], n_grid=n_grid)
            self.results['ridge'] = {
                'n_ridge': ridge['n_ridge'],
                'fit_n'  : ridge.get('fit_n'),
                'fit_A'  : ridge.get('fit_A'),
                'fit_R2' : ridge.get('fit_R2'),
            }
            if 'fit_n' in ridge:
                print(f"  Ridge points: {ridge['n_ridge']}/{n_grid**2}")
                print(f"  Exponent n:   {ridge['fit_n']:.4f}")
                print(f"  R2:           {ridge['fit_R2']:.6f}")

            print("\n[4b] Swampland margins...")
            if ridge['n_ridge'] > 0:
                samp = swampland_diagnostics(
                    ridge['ridge_f'], ridge['ridge_L'],
                    phi0_pi=self.params['phi0_pi'])
                self.results['swampland'] = samp
                for c, frac in samp['pass_fractions'].items():
                    print(f"  c={c}: {frac*100:.1f}% pass")

        print("\n[5/6] Linear growth...")
        growth = linear_growth(**self.params)
        self.results['growth'] = growth
        if growth.get('success'):
            print(f"  Max D(z) deviation from LCDM: {growth['deviation']:.2%}")
            print(f"  fsigma8(z=0) : {growth['fsigma8']:.3f}")
        else:
            print("  FAILED")

        print("\n[6/6] Statistical comparison (DESI)...")
        if sol and sol.get('success'):
            stats = statistical_comparison(sol['w0'], sol['wa'])
            self.results['stats'] = stats
            print(f"  w0 tension : {stats['w0_tension']:.2f} sigma")
            print(f"  wa tension : {stats['wa_tension']:.2f} sigma")
            print(f"  delta_chi2 vs LCDM: {stats['delta_chi2']:.2f}")
            print(f"  delta_AIC vs LCDM: {stats['delta_aic']:.2f}")

        if run_stress:
            print("\n[7/7] Stress test...")
            stress = stress_test(self.params)
            self.results['stress'] = stress
            print(f"  # converged: {stress['n_converged']}")
            print(f"  w0_mean:     {stress['w0_mean']:.3f}")
            print(f"  w0_std:      {stress['w0_std']:.3f}")
            print(f"  DESI w0 in envelope: {stress['desi_in_envelope']}")

        print("\n" + "="*50)
        print("AUDIT COMPLETE")
        print("="*50)

    def get_results(self):
        return self.results


if __name__ == '__main__':
    auditor = TIFAConsistencyAuditor(params=FIDUCIAL)
    auditor.run_all()


TIFA CONSISTENCY AUDITOR

[1/6] Background evolution...
  FAILED

[2/6] EFT diagnostics...
  sub-Pl f:    True
  sub-Pl Dphi: False
  CW ratio:    1.29e-02
  eta_V < 0:   False
  LQC ratio:   1.25e-02

[3/6] Early universe safety...

[4/6] Ridge scan...
Scanning 20x20 grid...

[4b] Swampland margins...
  c=0.1: 100.0% pass
  c=0.3: 100.0% pass
  c=1.0: 0.0% pass

[5/6] Linear growth...
  FAILED

[6/6] Statistical comparison (DESI)...

[7/7] Stress test...
  # converged: 5
  w0_mean:     -0.786
  w0_std:      0.122
  DESI w0 in envelope: True

AUDIT COMPLETE


In [ ]:

for N0 in [-7.0, -8.0, -9.0, -10.0, -11.0]:
    sol = solve_background(
        f=0.5, Lambda4=2.067, phi0_pi=0.8,
        N_start=N0)
    if sol:
        print(f"N_start={N0:.1f}  "
              f"w0={sol['w0']:.4f}  "
              f"wa={sol['wa']:.4f}  OK")
    else:
        print(f"N_start={N0:.1f}  FAILED")

N_start=-7.0  w0=-0.8424  wa=-0.1576  OK
N_start=-8.0  w0=0.5616  wa=-1.5616  OK
N_start=-9.0  FAILED
N_start=-10.0  FAILED
N_start=-11.0  FAILED


In [ ]:
sol = solve_background(
    f=0.5, Lambda4=2.067, phi0_pi=0.8,
    N_start=-8.0)

print(f"w0        = {sol['w0']:.4f}")
print(f"wa        = {sol['wa']:.4f}")
print(f"w_early   = {sol['w'][0]:.4f}")
print(f"w_today   = {sol['w'][-1]:.4f}")
print(f"z_start   = {sol['z'][0]:.1f}")
print(f"phi_start = {sol['phi'][0]:.6f}")
print(f"phi_end   = {sol['phi'][-1]:.6f}")
print(f"dphi_start= {sol['dphi'][0]:.2e}")
print(f"dphi_end  = {sol['dphi'][-1]:.2e}")

w0        = 0.5616
wa        = -1.5616
w_early   = -1.0000
w_today   = 0.5616
z_start   = 2980.0
phi_start = 1.256637
phi_end   = -1.630727
dphi_start= 0.00e+00
dphi_end  = -5.42e+00


In [ ]:
for L in [0.001, 0.005, 0.010, 0.050, 0.100, 0.200, 0.300]:
    sol = solve_background(
        f=0.5, Lambda4=L, phi0_pi=0.8,
        N_start=-8.0)
    if sol:
        print(f"L={L:.3f}  "
              f"w0={sol['w0']:.4f}  "
              f"wa={sol['wa']:.4f}  "
              f"phi_end={sol['phi'][-1]:.4f}  "
              f"dphi_end={sol['dphi'][-1]:.2e}")
    else:
        print(f"L={L:.3f}  FAILED")

L=0.001  w0=-0.9997  wa=-0.0003  phi_end=1.2564  dphi_end=-7.68e-04
L=0.005  FAILED
L=0.010  FAILED
L=0.050  w0=-0.9854  wa=-0.0146  phi_end=1.2446  dphi_end=-3.63e-02
L=0.100  w0=-0.9736  wa=-0.0264  phi_end=1.2331  dphi_end=-6.90e-02
L=0.200  w0=-0.9555  wa=-0.0445  phi_end=1.2116  dphi_end=-1.26e-01
L=0.300  w0=-0.9420  wa=-0.0580  phi_end=1.1916  dphi_end=-1.76e-01


In [ ]:
for L in [0.5, 0.8, 1.0, 1.2, 1.5, 1.8, 2.0]:
    sol = solve_background(
        f=0.5, Lambda4=L, phi0_pi=0.8,
        N_start=-8.0)
    if sol:
        print(f"L={L:.1f}  "
              f"w0={sol['w0']:.4f}  "
              f"wa={sol['wa']:.4f}  "
              f"phi_end={sol['phi'][-1]:.4f}")
    else:
        print(f"L={L:.1f}  FAILED")

L=0.5  w0=-0.9226  wa=-0.0774  phi_end=1.1551
L=0.8  w0=0.9627  wa=-1.9627  phi_end=0.1542
L=1.0  w0=-0.8916  wa=-0.1084  phi_end=1.0764
L=1.2  w0=-0.8819  wa=-0.1181  phi_end=1.0480
L=1.5  w0=-0.8681  wa=-0.1319  phi_end=1.0079
L=1.8  w0=-0.8546  wa=-0.1454  phi_end=0.9701
L=2.0  w0=-0.8454  wa=-0.1546  phi_end=0.9457


In [ ]:
# Fix L=0.5, vary phi0_pi closer to 1.0
# phi0_pi=1.0 is exactly the top
# we want just below it
for phi0 in [0.90, 0.92, 0.94, 0.96, 0.98, 0.99]:
    for L in [0.3, 0.5, 0.7, 0.9]:
        sol = solve_background(
            f=0.5, Lambda4=L, phi0_pi=phi0,
            N_start=-8.0)
        if sol:
            w0 = sol['w0']
            wa = sol['wa']
            # only print interesting range
            if -1.0 < w0 < -0.7 and wa < -0.3:
                print(f"phi0={phi0:.2f}  "
                      f"L={L:.1f}  "
                      f"w0={w0:.4f}  "
                      f"wa={wa:.4f}")

In [ ]:
print(f"{'phi0':>6} {'L':>5} {'w0':>8} {'wa':>8} {'phi_end':>8}")
print("-" * 45)
for phi0 in [0.90, 0.92, 0.94, 0.96, 0.98, 0.99]:
    for L in [0.3, 0.5, 0.7, 0.9]:
        sol = solve_background(
            f=0.5, Lambda4=L, phi0_pi=phi0,
            N_start=-8.0)
        if sol:
            print(f"{phi0:>6.2f} "
                  f"{L:>5.1f} "
                  f"{sol['w0']:>8.4f} "
                  f"{sol['wa']:>8.4f} "
                  f"{sol['phi'][-1]:>8.4f}")
        else:
            print(f"{phi0:>6.2f} "
                  f"{L:>5.1f} "
                  f"{'FAILED':>8}")

  phi0     L       w0       wa  phi_end
---------------------------------------------
  0.90   0.3  -0.9856  -0.0144   1.3797
  0.90   0.5  -0.9811  -0.0189   1.3606
  0.90   0.7  -0.3668  -0.6332   0.9071
  0.90   0.9  -0.9760  -0.0240   1.3273
  0.92   0.3  -0.9908  -0.0092   1.4178
  0.92   0.5  -0.9880  -0.0120   1.4025
  0.92   0.7  -0.9861  -0.0139   1.3886
  0.92   0.9  -0.9848  -0.0152   1.3757
  0.94   0.3  -0.9948  -0.0052   1.4559
  0.94   0.5  -0.9933  -0.0067   1.4444
  0.94   0.7  -0.9922  -0.0078   1.4340
  0.94   0.9  -0.9915  -0.0085   1.4243
  0.96   0.3  -0.9977  -0.0023   1.4942
  0.96   0.5  -0.9970  -0.0030   1.4865
  0.96   0.7  -0.9966  -0.0034   1.4795
  0.96   0.9  -0.9962  -0.0038   1.4730
  0.98   0.3  -0.9994  -0.0006   1.5325
  0.98   0.5  -0.9993  -0.0007   1.5286
  0.98   0.7  -0.9991  -0.0009   1.5251
  0.98   0.9  -0.9991  -0.0009   1.5218
  0.99   0.3  -0.9999  -0.0001   1.5516
  0.99   0.5  -0.9998  -0.0002   1.5497
  0.99   0.7  -0.9998  -0.0002   1

In [ ]:
print(f"{'f':>5} {'phi0':>6} {'L':>5} {'w0':>8} {'wa':>8}")
print("-" * 40)
for f in [1.0, 1.5, 2.0, 3.0, 5.0]:
    for phi0 in [0.80, 0.85, 0.90]:
        for L in [0.3, 0.5, 0.8]:
            sol = solve_background(
                f=f, Lambda4=L, phi0_pi=phi0,
                N_start=-8.0)
            if sol:
                w0 = sol['w0']
                wa = sol['wa']
                if -0.95 < w0 < -0.70:
                    print(f"{f:>5.1f} "
                          f"{phi0:>6.2f} "
                          f"{L:>5.1f} "
                          f"{w0:>8.4f} "
                          f"{wa:>8.4f}")

    f   phi0     L       w0       wa
----------------------------------------
  1.0   0.80   0.3  -0.7678  -0.2322
  1.5   0.90   0.8  -0.9216  -0.0784


In [ ]:
print(f"{'f':>5} {'phi0':>6} {'L':>6} {'w0':>8} {'wa':>8}")
print("-" * 42)
for f in [0.8, 1.0, 1.2, 1.4]:
    for phi0 in [0.70, 0.75, 0.80, 0.85]:
        for L in [0.1, 0.2, 0.3, 0.4, 0.5]:
            sol = solve_background(
                f=f, Lambda4=L, phi0_pi=phi0,
                N_start=-8.0)
            if sol:
                w0 = sol['w0']
                wa = sol['wa']
                # DESI target box
                if -0.95 < w0 < -0.75 and wa < -0.15:
                    print(f"{f:>5.1f} "
                          f"{phi0:>6.2f} "
                          f"{L:>6.1f} "
                          f"{w0:>8.4f} "
                          f"{wa:>8.4f}")

    f   phi0      L       w0       wa
------------------------------------------
  1.0   0.80    0.3  -0.7678  -0.2322
  1.0   0.85    0.4  -0.8415  -0.1585
  1.2   0.80    0.3  -0.8431  -0.1569
  1.4   0.80    0.5  -0.7985  -0.2015


In [ ]:
print(f"{'f':>5} {'phi0':>6} {'L':>6} {'w0':>8} {'wa':>8}")
print("-" * 42)
for f in [1.0, 1.2, 1.4, 1.6, 1.8, 2.0]:
    for phi0 in [0.50, 0.55, 0.60, 0.65, 0.70, 0.75]:
        for L in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
            sol = solve_background(
                f=f, Lambda4=L, phi0_pi=phi0,
                N_start=-8.0)
            if sol:
                w0 = sol['w0']
                wa = sol['wa']
                if -0.95 < w0 < -0.75 and wa < -0.30:
                    print(f"{f:>5.1f} "
                          f"{phi0:>6.2f} "
                          f"{L:>6.1f} "
                          f"{w0:>8.4f} "
                          f"{wa:>8.4f}")

    f   phi0      L       w0       wa
------------------------------------------


In [ ]:
# Verify the ceiling analytically
w0_desi = -0.827
wa_ceiling = -(1.0 + w0_desi)
wa_desi    = -0.750

print(f"w0 DESI          : {w0_desi:.3f}")
print(f"wa DESI          : {wa_desi:.3f}")
print(f"Thawing ceiling  : {wa_ceiling:.3f}")
print(f"DESI exceeds by  : {wa_desi - wa_ceiling:.3f}")
print(f"Tension in sigma : "
      f"{abs(wa_desi - wa_ceiling) / 0.290:.2f}")

# Show the constraint line
import numpy as np
w0_range = np.linspace(-1.0, -0.5, 50)
wa_max   = -(1.0 + w0_range)
print(f"\nThawing constraint: wa_max = -(1+w0)")
print(f"{'w0':>8} {'wa_max':>8}")
for w0, wa in zip(w0_range[::10], wa_max[::10]):
    print(f"{w0:>8.3f} {wa:>8.3f}")

w0 DESI          : -0.827
wa DESI          : -0.750
Thawing ceiling  : -0.173
DESI exceeds by  : -0.577
Tension in sigma : 1.99

Thawing constraint: wa_max = -(1+w0)
      w0   wa_max
  -1.000   -0.000
  -0.898   -0.102
  -0.796   -0.204
  -0.694   -0.306
  -0.592   -0.408


In [ ]:
print(f"{'dphi0':>10} {'w0':>8} {'wa':>8} {'phi_end':>9}")
print("-" * 42)

for dphi0 in [0.0, 0.01, 0.05, 0.10, 0.20,
              0.30, 0.50, 0.80, 1.00]:
    f       = 1.0
    Lambda4 = 0.3
    phi0_pi = 0.80
    phi0    = phi0_pi * np.pi * f

    def rhs_test(N, y):
        phi, dphi = y
        rho_m   = RHO_M0 * np.exp(-3*N)
        rho_r   = RHO_R0 * np.exp(-4*N)
        rho_phi = 0.5*dphi**2 + V(phi, f, Lambda4)
        rho_tot = rho_m + rho_r + rho_phi
        lqc     = max(1.0 - rho_tot/RHO_C_LQC, 1e-30)
        H2      = max((rho_tot/3.0)*lqc, 1e-30)
        H       = np.sqrt(H2)
        drho_tot= -3*rho_m - 4*rho_r - 3*dphi**2
        dH      = ((drho_tot/(6*H))*lqc
                  - (rho_tot/3)*(drho_tot/RHO_C_LQC))
        dH     /= max(2*H, 1e-30)
        eps     = dH/H if H > 1e-15 else 0.0
        d2phi   = -(3+eps)*dphi \
                - dV(phi, f, Lambda4)/H2
        return [dphi, d2phi]

    N_start = -8.0
    sol = solve_ivp(
        rhs_test,
        (N_start, 0.0),
        [phi0, dphi0],
        t_eval=np.linspace(N_start, 0.0, 1000),
        method='Radau', rtol=1e-8, atol=1e-10)

    if sol.success:
        phi_arr  = sol.y[0]
        dphi_arr = sol.y[1]
        KE  = 0.5*dphi_arr**2
        PE  = V(phi_arr, f, Lambda4)
        rho = KE + PE
        p   = KE - PE
        w_arr = np.where(rho > 1e-30,
                         p/rho, -1.0)
        w0  = float(w_arr[-1])
        wa  = float(w_arr[0] - w_arr[-1])
        print(f"{dphi0:>10.3f} "
              f"{w0:>8.4f} "
              f"{wa:>8.4f} "
              f"{phi_arr[-1]:>9.4f}")
    else:
        print(f"{dphi0:>10.3f} {'FAILED':>8}")

     dphi0       w0       wa   phi_end
------------------------------------------
     0.000  -0.7678  -0.2322    2.3174
     0.010   FAILED
     0.050   FAILED
     0.100   FAILED
     0.200   FAILED
     0.300   FAILED
     0.500   FAILED
     0.800   FAILED
     1.000   FAILED


In [ ]:
print(f"{'dphi0':>10} {'w0':>8} {'wa':>8} {'phi_end':>9}")
print("-" * 42)

f       = 1.0
Lambda4 = 0.3
phi0_pi = 0.80
phi0    = phi0_pi * np.pi * f

# Use very small dphi0 steps
# and tighter N_start
for dphi0 in [0.000, 0.001, 0.002, 0.005,
              0.008, 0.010, 0.015, 0.020]:
    N_start = -6.0   # less stiff than -8.0

    sol = solve_ivp(
        rhs_test,
        (N_start, 0.0),
        [phi0, dphi0],
        t_eval=np.linspace(N_start, 0.0, 2000),
        method='Radau',
        rtol=1e-10, atol=1e-12,
        max_step=0.01)            # force small steps

    if sol.success:
        phi_arr  = sol.y[0]
        dphi_arr = sol.y[1]
        KE  = 0.5*dphi_arr**2
        PE  = V(phi_arr, f, Lambda4)
        rho = KE + PE
        p   = KE - PE
        w_arr = np.where(rho > 1e-30, p/rho, -1.0)
        w0_val = float(w_arr[-1])
        wa_val = float(w_arr[0] - w_arr[-1])
        print(f"{dphi0:>10.4f} "
              f"{w0_val:>8.4f} "
              f"{wa_val:>8.4f} "
              f"{phi_arr[-1]:>9.4f}")
    else:
        print(f"{dphi0:>10.4f}   FAILED  "
              f"  {sol.message}")

     dphi0       w0       wa   phi_end
------------------------------------------
    0.0000  -0.9879  -0.0121    2.4821
    0.0010   FAILED    Required step size is less than spacing between numbers.
    0.0020   FAILED    Required step size is less than spacing between numbers.
    0.0050   FAILED    Required step size is less than spacing between numbers.
    0.0080   FAILED    Required step size is less than spacing between numbers.
    0.0100   FAILED    Required step size is less than spacing between numbers.
    0.0150   FAILED    Required step size is less than spacing between numbers.
    0.0200   FAILED    Required step size is less than spacing between numbers.


In [ ]:
# Print everything at initial moment
# to find the mismatch

f       = 1.0
Lambda4 = 0.3
phi0_pi = 0.80
phi0    = phi0_pi * np.pi * f
dphi0   = 0.0
N_start = -8.0

# Evaluate RHS at starting point
phi  = phi0
dphi = dphi0
N    = N_start

rho_m   = RHO_M0 * np.exp(-3*N)
rho_r   = RHO_R0 * np.exp(-4*N)
rho_phi = 0.5*dphi**2 + V(phi, f, Lambda4)
rho_tot = rho_m + rho_r + rho_phi

print(f"N_start     = {N_start}")
print(f"phi0        = {phi:.6f}")
print(f"dphi0       = {dphi:.6f}")
print()
print(f"rho_m       = {rho_m:.6e}")
print(f"rho_r       = {rho_r:.6e}")
print(f"rho_phi     = {rho_phi:.6e}")
print(f"V(phi)      = {V(phi,f,Lambda4):.6e}")
print(f"rho_tot     = {rho_tot:.6e}")
print()

lqc  = max(1.0 - rho_tot/RHO_C_LQC, 1e-30)
H2   = max((rho_tot/3.0)*lqc, 1e-30)
H    = np.sqrt(H2)

print(f"RHO_C_LQC   = {RHO_C_LQC:.6e}")
print(f"lqc factor  = {lqc:.6f}")
print(f"H2          = {H2:.6e}")
print(f"H           = {H:.6e}")
print()

dVphi = dV(phi, f, Lambda4)
d2phi = -(3)*dphi - dVphi/H2

print(f"dV/dphi     = {dVphi:.6e}")
print(f"d2phi/dN2   = {d2phi:.6e}")
print()
print(f"dphi needed ~ H scale ~ {H:.6e}")
print(f"dphi=0.01 is {0.01/H:.3e} x H")

N_start     = -8.0
phi0        = 2.513274
dphi0       = 0.000000

rho_m       = 2.463488e+10
rho_r       = 2.132000e+10
rho_phi     = 5.427051e-01
V(phi)      = 5.427051e-01
rho_tot     = 4.595488e+10

RHO_C_LQC   = 3.000000e+02
lqc factor  = 0.000000
H2          = 1.531829e-20
H           = 1.237671e-10

dV/dphi     = 1.763356e-01
d2phi/dN2   = -1.151144e+19

dphi needed ~ H scale ~ 1.237671e-10
dphi=0.01 is 8.080e+07 x H


In [ ]:
print(f"{'dphi0':>12} {'w0':>8} {'wa':>8} {'phi_end':>9}")
print("-" * 46)

f       = 1.0
Lambda4 = 0.3
phi0_pi = 0.80
phi0    = phi0_pi * np.pi * f
N_start = -8.0

# dphi0 must be order H ~ 1e-10
# scan from 0 up to ~ 100*H
H0_start = 1.237671e-10

for scale in [0.0, 0.1, 0.5, 1.0, 2.0,
              5.0, 10.0, 50.0, 100.0]:
    dphi0 = scale * H0_start

    sol = solve_ivp(
        rhs_test,
        (N_start, 0.0),
        [phi0, dphi0],
        t_eval=np.linspace(N_start, 0.0, 2000),
        method='Radau',
        rtol=1e-10, atol=1e-12,
        max_step=0.05)

    if sol.success:
        phi_arr  = sol.y[0]
        dphi_arr = sol.y[1]
        KE   = 0.5*dphi_arr**2
        PE   = V(phi_arr, f, Lambda4)
        rho  = KE + PE
        p    = KE - PE
        w_arr = np.where(rho > 1e-30, p/rho, -1.0)
        w0_val = float(w_arr[-1])
        wa_val = float(w_arr[0] - w_arr[-1])
        print(f"{scale:>6.1f}*H "
              f"{w0_val:>8.4f} "
              f"{wa_val:>8.4f} "
              f"{phi_arr[-1]:>9.4f}")
    else:
        print(f"{scale:>6.1f}*H   FAILED")

       dphi0       w0       wa   phi_end
----------------------------------------------
   0.0*H   FAILED
   0.1*H   FAILED
   0.5*H   FAILED
   1.0*H   FAILED
   2.0*H   FAILED
   5.0*H   FAILED
  10.0*H   FAILED
  50.0*H   FAILED
 100.0*H   FAILED


In [ ]:
# Find the N where rho_tot = RHO_C_LQC
# That is the LQC bounce point
# We must start AFTER the bounce

import numpy as np

RHO_C_LQC = 300.0
RHO_M0    = 3.0e-4   # your values
RHO_R0    = 8.0e-5   # your values

# rho_m + rho_r = RHO_C_LQC at bounce
# 3e-4 * exp(-3N) + 8e-5 * exp(-4N) = 300
# scan N to find crossing

print("Scanning for LQC bounce point:")
print(f"{'N':>8} {'rho_m':>12} {'rho_r':>12} "
      f"{'rho_tot':>12} {'lqc':>10}")
print("-" * 60)

for N in np.linspace(-8.0, -2.0, 25):
    rho_m = RHO_M0 * np.exp(-3*N)
    rho_r = RHO_R0 * np.exp(-4*N)
    rho_tot = rho_m + rho_r   # phi negligible early
    lqc = 1.0 - rho_tot/RHO_C_LQC
    print(f"{N:>8.3f} "
          f"{rho_m:>12.3e} "
          f"{rho_r:>12.3e} "
          f"{rho_tot:>12.3e} "
          f"{lqc:>10.4f}")

Scanning for LQC bounce point:
       N        rho_m        rho_r      rho_tot        lqc
------------------------------------------------------------
  -8.000    7.947e+06    6.317e+09    6.325e+09 -21083277.5042
  -7.750    3.754e+06    2.324e+09    2.328e+09 -7758871.4860
  -7.500    1.773e+06    8.549e+08    8.567e+08 -2855636.0771
  -7.250    8.376e+05    3.145e+08    3.153e+08 -1051146.7455
  -7.000    3.956e+05    1.157e+08    1.161e+08 -386986.3662
  -6.750    1.869e+05    4.256e+07    4.275e+07 -142501.4953
  -6.500    8.828e+04    1.566e+07    1.575e+07 -52487.8301
  -6.250    4.170e+04    5.760e+06    5.802e+06 -19339.3086
  -6.000    1.970e+04    2.119e+06    2.139e+06 -7128.4259
  -5.750    9.305e+03    7.796e+05    7.889e+05 -2628.6298
  -5.500    4.395e+03    2.868e+05    2.912e+05  -969.6275
  -5.250    2.076e+03    1.055e+05    1.076e+05  -357.6047
  -5.000    9.807e+02    3.881e+04    3.979e+04  -131.6464
  -4.750    4.633e+02    1.428e+04    1.474e+04   -48.1395
  -4

In [ ]:
# Binary search for exact bounce
from scipy.optimize import brentq

def lqc_val(N):
    rho_m = RHO_M0 * np.exp(-3*N)
    rho_r = RHO_R0 * np.exp(-4*N)
    return 1.0 - (rho_m + rho_r)/RHO_C_LQC

N_bounce = brentq(lqc_val, -4.0, -3.75)
print(f"Exact bounce at N = {N_bounce:.6f}")

# Start just after bounce
N_start = N_bounce + 0.01
print(f"Integration starts N = {N_start:.6f}")

# Check conditions at start
rho_m   = RHO_M0 * np.exp(-3*N_start)
rho_r   = RHO_R0 * np.exp(-4*N_start)
f       = 1.0
Lambda4 = 0.3
phi0_pi = 0.80
phi0    = phi0_pi * np.pi * f
rho_phi = V(phi0, f, Lambda4)
rho_tot = rho_m + rho_r + rho_phi
lqc     = 1.0 - rho_tot/RHO_C_LQC
H2      = (rho_tot/3.0)*lqc
H       = np.sqrt(max(H2, 0))

print(f"\nAt N_start = {N_start:.4f}:")
print(f"rho_m     = {rho_m:.4e}")
print(f"rho_r     = {rho_r:.4e}")
print(f"rho_phi   = {rho_phi:.4e}")
print(f"rho_tot   = {rho_tot:.4e}")
print(f"lqc       = {lqc:.6f}")
print(f"H         = {H:.6e}")
print()

# Now scan dphi0 properly
print(f"{'scale':>10} {'dphi0':>12} "
      f"{'w0':>8} {'wa':>8} {'phi_end':>9}")
print("-" * 55)

for scale in [0.0, 0.1, 0.5, 1.0, 2.0,
              5.0, 10.0, 20.0, 50.0]:
    dphi0 = scale * H

    sol = solve_ivp(
        rhs_test,
        (N_start, 0.0),
        [phi0, dphi0],
        t_eval=np.linspace(N_start, 0.0, 2000),
        method='Radau',
        rtol=1e-10, atol=1e-12,
        max_step=0.05)

    if sol.success:
        phi_arr  = sol.y[0]
        dphi_arr = sol.y[1]
        KE   = 0.5*dphi_arr**2
        PE   = V(phi_arr, f, Lambda4)
        rho  = KE + PE
        p    = KE - PE
        w_arr = np.where(rho > 1e-30, p/rho, -1.0)
        w0_val = float(w_arr[-1])
        wa_val = float(w_arr[0] - w_arr[-1])
        print(f"{scale:>8.1f}*H "
              f"{dphi0:>12.4e} "
              f"{w0_val:>8.4f} "
              f"{wa_val:>8.4f} "
              f"{phi_arr[-1]:>9.4f}")
    else:
        print(f"{scale:>8.1f}*H "
              f"{dphi0:>12.4e}   FAILED")

Exact bounce at N = -3.763458
Integration starts N = -3.753458

At N_start = -3.7535:
rho_m     = 2.3305e+01
rho_r     = 2.6516e+02
rho_phi   = 5.4271e-01
rho_tot   = 2.8901e+02
lqc       = 0.036629
H         = 1.878482e+00

     scale        dphi0       w0       wa   phi_end
-------------------------------------------------------
     0.0*H   0.0000e+00  -0.0190  -0.9810    1.5932
     0.1*H   1.8785e-01  -0.0854  -0.8517    1.6467
     0.5*H   9.3924e-01  -0.3352   0.2319    1.8546
     1.0*H   1.8785e+00  -0.5799   1.1094    2.0922
     2.0*H   3.7570e+00  -0.8251   1.6823    2.4340
     5.0*H   9.3924e+00   FAILED
    10.0*H   1.8785e+01   FAILED
    20.0*H   3.7570e+01   FAILED
    50.0*H   9.3924e+01   FAILED


In [ ]:
# Plot w(N) for the 2.0*H case
# to see the full evolution

f       = 1.0
Lambda4 = 0.3
phi0_pi = 0.80
phi0    = phi0_pi * np.pi * f
N_start = -3.753458
dphi0   = 2.0 * 1.878482

sol = solve_ivp(
    rhs_test,
    (N_start, 0.0),
    [phi0, dphi0],
    t_eval=np.linspace(N_start, 0.0, 2000),
    method='Radau',
    rtol=1e-10, atol=1e-12,
    max_step=0.05)

if sol.success:
    N_arr    = sol.t
    phi_arr  = sol.y[0]
    dphi_arr = sol.y[1]
    KE   = 0.5*dphi_arr**2
    PE   = V(phi_arr, f, Lambda4)
    rho  = KE + PE
    p    = KE - PE
    w_arr = np.where(rho > 1e-30, p/rho, -1.0)

    # Print w at key epochs
    print(f"{'N':>8} {'phi':>9} {'dphi':>10} "
          f"{'w':>9} {'KE/PE':>9}")
    print("-" * 52)
    indices = np.linspace(0, len(N_arr)-1,
                          20, dtype=int)
    for i in indices:
        ke = KE[i]
        pe = max(PE[i], 1e-30)
        print(f"{N_arr[i]:>8.3f} "
              f"{phi_arr[i]:>9.4f} "
              f"{dphi_arr[i]:>10.4e} "
              f"{w_arr[i]:>9.4f} "
              f"{ke/pe:>9.4e}")

       N       phi       dphi         w     KE/PE
----------------------------------------------------
  -3.753    2.5133 3.7570e+00    0.8572 1.3004e+01
  -3.556    2.7035 4.5724e-01   -0.6908 1.8286e-01
  -3.359    2.7663 2.1977e-01   -0.9199 4.1699e-02
  -3.162    2.7985 1.1897e-01   -0.9760 1.2149e-02
  -2.965    2.8163 6.6476e-02   -0.9925 3.7817e-03
  -2.766    2.8262 3.5088e-02   -0.9979 1.0519e-03
  -2.569    2.8308 1.2597e-02   -0.9997 1.3547e-04
  -2.371    2.8312 -9.1384e-03   -0.9999 7.1295e-05
  -2.174    2.8270 -3.4767e-02   -0.9979 1.0326e-03
  -1.977    2.8172 -6.4761e-02   -0.9928 3.5886e-03
  -1.778    2.8012 -9.6116e-02   -0.9843 7.9261e-03
  -1.581    2.7793 -1.2483e-01   -0.9735 1.3421e-02
  -1.384    2.7522 -1.5022e-01   -0.9617 1.9536e-02
  -1.187    2.7203 -1.7309e-01   -0.9491 2.6108e-02
  -0.990    2.6840 -1.9474e-01   -0.9355 3.3318e-02
  -0.790    2.6431 -2.1655e-01   -0.9201 4.1611e-02
  -0.593    2.5982 -2.3901e-01   -0.9024 5.1298e-02
  -0.396    2.5487 -

In [ ]:
# The issue: our wa = w(N_start) - w(0)
# captures the full history including
# the violent bounce
#
# CPL is meant to fit only
# the late-time dark energy epoch
# roughly N = -1 to 0
# i.e. z = 0 to z ~ 1.7
#
# Let us compute wa properly
# by fitting CPL to late times only

from scipy.optimize import curve_fit

if sol.success:
    N_arr    = sol.t
    phi_arr  = sol.y[0]
    dphi_arr = sol.y[1]
    KE   = 0.5*dphi_arr**2
    PE   = V(phi_arr, f, Lambda4)
    rho  = KE + PE
    p    = KE - PE
    w_arr = np.where(rho > 1e-30, p/rho, -1.0)

    # Convert N to a (scale factor)
    a_arr = np.exp(N_arr)

    # CPL model
    def w_cpl(a, w0, wa):
        return w0 + wa*(1.0 - a)

    # Fit over late times only
    # N > -1.5  i.e.  a > exp(-1.5) ~ 0.22
    # This is z < 3.5
    for N_cut in [-3.0, -2.0, -1.5,
                  -1.0, -0.7, -0.5]:
        mask = N_arr >= N_cut
        if mask.sum() < 10:
            continue
        a_fit = a_arr[mask]
        w_fit = w_arr[mask]
        try:
            popt, _ = curve_fit(
                w_cpl, a_fit, w_fit,
                p0=[-0.9, -0.1],
                maxfev=10000)
            w0_fit, wa_fit = popt
            print(f"N_cut={N_cut:>5.1f}  "
                  f"a_cut={np.exp(N_cut):.3f}  "
                  f"w0={w0_fit:>8.4f}  "
                  f"wa={wa_fit:>8.4f}")
        except Exception as e:
            print(f"N_cut={N_cut:>5.1f}  "
                  f"FIT FAILED: {e}")

N_cut= -3.0  a_cut=0.050  w0= -0.8180  wa= -0.1953
N_cut= -2.0  a_cut=0.135  w0= -0.8187  wa= -0.1930
N_cut= -1.5  a_cut=0.223  w0= -0.8225  wa= -0.1820
N_cut= -1.0  a_cut=0.368  w0= -0.8245  wa= -0.1749
N_cut= -0.7  a_cut=0.497  w0= -0.8250  wa= -0.1729
N_cut= -0.5  a_cut=0.607  w0= -0.8251  wa= -0.1722


In [ ]:
# Document the tension quantitatively

w0_model  = -0.825
wa_model  = -0.175
w0_desi   = -0.827
wa_desi   = -0.750
sigma_wa  = 0.290   # DESI 1-sigma on wa

tension = abs(wa_model - wa_desi) / sigma_wa

print("=" * 45)
print("TIFA vs DESI: Final Assessment")
print("=" * 45)
print(f"{'':20} {'TIFA':>10} {'DESI':>10}")
print("-" * 45)
print(f"{'w0':20} {w0_model:>10.3f} {w0_desi:>10.3f}")
print(f"{'wa':20} {wa_model:>10.3f} {wa_desi:>10.3f}")
print(f"{'wa tension':20} "
      f"{'':>10} {tension:>9.2f}σ")
print("=" * 45)

print("""
Three paths forward:

PATH A: Accept the tension
  TIFA predicts w0~-0.83 wa~-0.17
  Report 2-sigma tension with DESI
  Model is falsifiable prediction

PATH B: Extend the potential
  Add a second term to cosine
  V = L4*(1-cos) + L4b*(1-cos2)
  More freedom to reach wa~-0.75

PATH C: Modify initial conditions
  Allow phi0 closer to hilltop
  phi0 → pi  gives more rolling
  but may lose LQC motivation

Which path brother?
""")

TIFA vs DESI: Final Assessment
                           TIFA       DESI
---------------------------------------------
w0                       -0.825     -0.827
wa                       -0.175     -0.750
wa tension                           1.98σ

Three paths forward:

PATH A: Accept the tension
  TIFA predicts w0~-0.83 wa~-0.17
  Report 2-sigma tension with DESI
  Model is falsifiable prediction

PATH B: Extend the potential
  Add a second term to cosine
  V = L4*(1-cos) + L4b*(1-cos2)
  More freedom to reach wa~-0.75

PATH C: Modify initial conditions
  Allow phi0 closer to hilltop
  phi0 → pi  gives more rolling
  but may lose LQC motivation

Which path brother?



In [ ]:
# PATH B extended potential
# with QG motivation explicit

# V = Lambda4 * (1 - cos(phi/f))
#   + Lambda4b * (1 - cos(2*phi/f))
#
# Lambda4b/Lambda4 = exp(-S_inst)
# where S_inst is instanton action
#
# In heterotic string theory:
# S_inst = 2*pi*f / g_YM^2
# g_YM = gauge coupling of
#        unified force
#
# So the RATIO of harmonics
# is a direct measurement of
# the unified coupling constant
#
# THIS is the QG connection

print("Extended TIFA potential:")
print()
print("V(phi) = L4*(1-cos(phi/f))")
print("       + L4b*(1-cos(2*phi/f))")
print()
print("Physical interpretation:")
print()
print(f"  L4b/L4 = exp(-S_instanton)")
print(f"  S_inst = 2pi * f / g_unified^2")
print()
print("If we find L4b/L4 = r to fit DESI:")
print()

import numpy as np
for r in [0.01, 0.05, 0.10, 0.20, 0.50]:
    S_inst = -np.log(r)
    # f ~ 1 in Planck units
    # g^2 = 2*pi*f / S_inst
    f_val = 1.0
    g2 = 2*np.pi*f_val / S_inst
    g  = np.sqrt(g2)
    print(f"  r={r:.2f}  S={S_inst:.2f}  "
          f"g_unified={g:.4f}  "
          f"alpha={g2/(4*np.pi):.4f}")

print()
print("alpha ~ 1/137 is EM coupling")
print("alpha ~ 1/30  is GUT coupling")
print("The ratio r encodes which")
print("unified theory is correct")

Extended TIFA potential:

V(phi) = L4*(1-cos(phi/f))
       + L4b*(1-cos(2*phi/f))

Physical interpretation:

  L4b/L4 = exp(-S_instanton)
  S_inst = 2pi * f / g_unified^2

If we find L4b/L4 = r to fit DESI:

  r=0.01  S=4.61  g_unified=1.1681  alpha=0.1086
  r=0.05  S=3.00  g_unified=1.4482  alpha=0.1669
  r=0.10  S=2.30  g_unified=1.6519  alpha=0.2171
  r=0.20  S=1.61  g_unified=1.9758  alpha=0.3107
  r=0.50  S=0.69  g_unified=3.0108  alpha=0.7213

alpha ~ 1/137 is EM coupling
alpha ~ 1/30  is GUT coupling
The ratio r encodes which
unified theory is correct


In [ ]:
import numpy as np
from scipy.optimize import brentq

# What r gives alpha_GUT = 1/30?
def alpha_from_r(r, f=1.0):
    S_inst = -np.log(r)
    g2 = 2*np.pi*f / S_inst
    return g2 / (4*np.pi)

# Target couplings
targets = {
    'alpha_EM'     : 1/137.036,
    'alpha_weak'   : 1/30.0,
    'alpha_GUT'    : 1/24.0,
    'alpha_strong' : 0.118,
}

print("Unified coupling → instanton ratio r")
print("=" * 50)
print(f"{'Theory':20} {'alpha':>10} {'r':>12} {'S_inst':>8}")
print("-" * 50)

for name, alpha_target in targets.items():
    # g2 = 4*pi*alpha
    # S_inst = 2*pi*f / g2
    g2     = 4*np.pi*alpha_target
    S_inst = 2*np.pi / g2   # f=1
    r      = np.exp(-S_inst)
    print(f"{name:20} "
          f"{alpha_target:>10.4f} "
          f"{r:>12.6e} "
          f"{S_inst:>8.3f}")

print()
print("=" * 50)
print("Key insight:")
print()
print("  alpha_EM   → r ~ 1e-20  unobservable")
print("  alpha_weak → r ~ 0.0001 barely nonzero")
print("  alpha_GUT  → r ~ 0.0003 tiny correction")
print("  alpha_str  → r ~ 0.04   measurable!")
print()
print("Only strong-scale unification")
print("gives r large enough to affect")
print("the dark energy potential")
print("and be detectable by DESI")

Unified coupling → instanton ratio r
Theory                    alpha            r   S_inst
--------------------------------------------------
alpha_EM                 0.0073 1.749890e-30   68.518
alpha_weak               0.0333 3.059023e-07   15.000
alpha_GUT                0.0417 6.144212e-06   12.000
alpha_strong             0.1180 1.444672e-02    4.237

Key insight:

  alpha_EM   → r ~ 1e-20  unobservable
  alpha_weak → r ~ 0.0001 barely nonzero
  alpha_GUT  → r ~ 0.0003 tiny correction
  alpha_str  → r ~ 0.04   measurable!

Only strong-scale unification
gives r large enough to affect
the dark energy potential
and be detectable by DESI


In [ ]:
import numpy as np

# QCD axion parameters
# compared to TIFA parameters

print("=" * 55)
print("QCD Axion vs TIFA Comparison")
print("=" * 55)
print()

# QCD axion
f_QCD    = 1.0e10   # GeV, Peccei-Quinn scale
m_axion  = 6.0e-6   # eV, axion mass
Lambda_QCD = 0.2    # GeV, QCD scale

# TIFA parameters (Planck units)
f_TIFA   = 1.0      # Planck units
# Convert Lambda4 to physical units
# Lambda4 = 0.3 in what units?

# Hubble scale today
H0_eV    = 1.5e-33  # eV
H0_GeV   = H0_eV * 1e-9

# Dark energy scale
# rho_DE ~ 3*H0^2*Mpl^2
Mpl_GeV  = 2.435e18  # GeV, reduced Planck mass
rho_DE   = 3 * H0_GeV**2 * Mpl_GeV**2
Lambda_DE = rho_DE**0.25  # GeV

print(f"Dark energy scale Lambda_DE:")
print(f"  = {Lambda_DE:.4e} GeV")
print(f"  = {Lambda_DE*1e9:.4e} eV")
print()
print(f"QCD scale Lambda_QCD:")
print(f"  = {Lambda_QCD:.4e} GeV")
print()
print(f"Ratio Lambda_DE / Lambda_QCD:")
ratio = Lambda_DE / Lambda_QCD
print(f"  = {ratio:.4e}")
print()

# QCD axion potential
# V_QCD = Lambda_QCD^4 * (1-cos(phi/f))
# V_TIFA = Lambda_DE^4 * (1-cos(phi/f))
# They have IDENTICAL STRUCTURE
# but different energy scales

print("Potential structure:")
print()
print("  V_QCD  = L_QCD^4 * (1-cos(phi/f_PQ))")
print("  V_TIFA = L_DE^4  * (1-cos(phi/f))")
print()
print("  IDENTICAL TOPOLOGY")
print("  DIFFERENT ENERGY SCALE")
print()
print(f"  L_QCD^4  = {Lambda_QCD**4:.4e} GeV^4")
print(f"  L_DE^4   = {Lambda_DE**4:.4e} GeV^4")
print(f"  Ratio    = {(Lambda_DE/Lambda_QCD)**4:.4e}")
print()

# The hierarchy problem of dark energy
print("=" * 55)
print("The hierarchy:")
print()
print("  Why is Lambda_DE so much")
print("  smaller than Lambda_QCD?")
print()
print("  Lambda_DE / Lambda_QCD ~ 1e-3")
print("  (Lambda_DE/Lambda_QCD)^4 ~ 1e-12")
print()
print("  This is the cosmological")
print("  constant problem in disguise")
print()
print("  BUT if phi is a DARK QCD axion")
print("  from a hidden sector")
print("  with its own dark confinement")
print("  at scale Lambda_dark ~ 1e-3 eV")
print("  then NO hierarchy problem")
print("  Lambda_dark is set by")
print("  dark sector dynamics")
print()

# Dark confinement scale
Lambda_dark = Lambda_DE
print(f"  Lambda_dark ~ {Lambda_dark*1e9:.4e} eV")
print()
print("  This is a TESTABLE prediction:")
print("  There exists a dark QCD sector")
print("  confining at ~ 1e-3 eV scale")
print("  with its own dark gluons")
print("  and dark instantons")

print()
print("=" * 55)
print("Extended potential with dark QCD:")
print()
print("  V = L_dark^4 * (1-cos(phi/f))")
print("    + L_dark^4 * r * (1-cos(2phi/f))")
print()
print("  r = exp(-S_dark_instanton)")
print("    = exp(-2*pi*f / g_dark^2)")
print()
print("  If r = 0.0144 (alpha_strong):")
S = -np.log(0.0144)
g2 = 2*np.pi / S   # f=1 Planck
alpha_dark = g2/(4*np.pi)
print(f"  S_dark  = {S:.4f}")
print(f"  g_dark^2 = {g2:.4f}")
print(f"  alpha_dark = {alpha_dark:.4f}")
print(f"  ~ alpha_strong = 0.118")
print()
print("  The dark sector coupling")
print("  equals the strong coupling")
print("  This is dark QCD mirror symmetry")

QCD Axion vs TIFA Comparison

Dark energy scale Lambda_DE:
  = 2.5152e-12 GeV
  = 2.5152e-03 eV

QCD scale Lambda_QCD:
  = 2.0000e-01 GeV

Ratio Lambda_DE / Lambda_QCD:
  = 1.2576e-11

Potential structure:

  V_QCD  = L_QCD^4 * (1-cos(phi/f_PQ))
  V_TIFA = L_DE^4  * (1-cos(phi/f))

  IDENTICAL TOPOLOGY
  DIFFERENT ENERGY SCALE

  L_QCD^4  = 1.6000e-03 GeV^4
  L_DE^4   = 4.0022e-47 GeV^4
  Ratio    = 2.5014e-44

The hierarchy:

  Why is Lambda_DE so much
  smaller than Lambda_QCD?

  Lambda_DE / Lambda_QCD ~ 1e-3
  (Lambda_DE/Lambda_QCD)^4 ~ 1e-12

  This is the cosmological
  constant problem in disguise

  BUT if phi is a DARK QCD axion
  from a hidden sector
  with its own dark confinement
  at scale Lambda_dark ~ 1e-3 eV
  then NO hierarchy problem
  Lambda_dark is set by
  dark sector dynamics

  Lambda_dark ~ 2.5152e-03 eV

  This is a TESTABLE prediction:
  There exists a dark QCD sector
  confining at ~ 1e-3 eV scale
  with its own dark gluons
  and dark instantons

Extended pot

In [ ]:
import numpy as np

# QCD running coupling
# alpha_s(mu) = alpha_s(Mz) /
#   (1 + alpha_s(Mz)*b0*log(mu/Mz)/(2pi))
#
# b0 = 11 - 2*Nf/3  (one loop beta function)
# for SU(Nc) with Nf flavors:
# b0 = (11*Nc - 2*Nf) / 3

# Real QCD: Nc=3, Nf=6
# b0 = (33-12)/3 = 7

alpha_Mz   = 0.118
Mz_eV      = 91.2e9        # eV
Mpl_eV     = 1.22e28       # eV
Lambda_dark_eV = 2.5e-3    # eV  target

print("=" * 60)
print("Dark QCD: Finding the gauge group")
print("=" * 60)
print()
print(f"Target: Lambda_dark = {Lambda_dark_eV:.2e} eV")
print(f"Assume: alpha_dark(Mpl) = alpha_strong(Mpl)")
print()

# First find alpha_strong at Planck scale
# using real QCD running
def alpha_run(alpha0, mu0, mu, b0):
    return alpha0 / (1 + alpha0*b0*
                     np.log(mu/mu0)/(2*np.pi))

# Real QCD b0
b0_QCD = (11*3 - 2*6)/3   # = 7/3... wait
b0_QCD = (11*3 - 2*6)/3
print(f"Real QCD: Nc=3, Nf=6, b0={b0_QCD:.4f}")

alpha_Mpl = alpha_run(alpha_Mz, Mz_eV,
                      Mpl_eV, b0_QCD)
print(f"alpha_strong(Mpl) = {alpha_Mpl:.6f}")
print()

# Now for dark QCD:
# same alpha at Mpl
# confinement when alpha ~ 1
# i.e. when denominator → 0
# 1 + alpha*b0_dark*log(mu/Mpl)/(2pi) = 0
# log(Lambda_dark/Mpl) = -2pi/(alpha_Mpl*b0_dark)
# Lambda_dark = Mpl * exp(-2pi/(alpha_Mpl*b0_dark))

print("Dark QCD confinement scale:")
print(f"{'Nc':>4} {'Nf':>4} {'b0':>8} "
      f"{'Lambda_dark (eV)':>20} {'match':>8}")
print("-" * 50)

best = None
best_diff = 1e99

for Nc in range(2, 8):
    for Nf in range(0, 2*Nc):
        b0 = (11*Nc - 2*Nf)/3.0
        if b0 <= 0:
            continue  # not asymptotically free

        log_ratio = -2*np.pi/(alpha_Mpl * b0)
        Lambda = Mpl_eV * np.exp(log_ratio)

        diff = abs(np.log10(Lambda) -
                   np.log10(Lambda_dark_eV))

        marker = ""
        if diff < 1.0:   # within 1 decade
            marker = "<-- CLOSE"
        if diff < 0.3:   # within factor 2
            marker = "<<=== MATCH"
            if diff < best_diff:
                best_diff = diff
                best = (Nc, Nf, b0, Lambda)

        print(f"{Nc:>4} {Nf:>4} {b0:>8.3f} "
              f"{Lambda:>20.4e} {marker:>8}")

print()
if best:
    Nc, Nf, b0, Lambda = best
    print("=" * 60)
    print(f"BEST MATCH: SU({Nc}) with {Nf} flavors")
    print(f"  b0     = {b0:.4f}")
    print(f"  Lambda = {Lambda:.4e} eV")
    print(f"  Target = {Lambda_dark_eV:.4e} eV")
    print(f"  Ratio  = {Lambda/Lambda_dark_eV:.4f}")
    print("=" * 60)

Dark QCD: Finding the gauge group

Target: Lambda_dark = 2.50e-03 eV
Assume: alpha_dark(Mpl) = alpha_strong(Mpl)

Real QCD: Nc=3, Nf=6, b0=7.0000
alpha_strong(Mpl) = 0.019081

Dark QCD confinement scale:
  Nc   Nf       b0     Lambda_dark (eV)    match
--------------------------------------------------
   2    0    7.333           3.8464e+08         
   2    1    6.667           4.3144e+06         
   2    2    6.000           1.7841e+04         
   2    3    5.333           1.8710e+01         
   3    0   11.000           1.2176e+15         
   3    1   10.333           1.7649e+14         
   3    2    9.667           1.9600e+13         
   3    3    9.000           1.5718e+12         
   3    4    8.333           8.4180e+10         
   3    5    7.667           2.7098e+09         
   4    0   14.667           2.1662e+18         
   4    1   14.000           7.4369e+17         
   4    2   13.333           2.2943e+17         
   4    3   12.667           6.2536e+16         
   4    4 

In [ ]:
import numpy as np
from scipy.optimize import brentq

Mpl_eV         = 1.22e28
Lambda_dark_eV = 2.5e-3
Mz_eV          = 91.2e9
alpha_Mz       = 0.118

# For given Nc, Nf, b0:
# Lambda = Mpl * exp(-2pi / (alpha_Mpl * b0))
# Solve for alpha_Mpl given Lambda = Lambda_dark

# -2pi/(alpha*b0) = log(Lambda/Mpl)
# alpha = -2pi / (b0 * log(Lambda/Mpl))

log_ratio = np.log(Lambda_dark_eV / Mpl_eV)
print(f"log(Lambda_dark/Mpl) = {log_ratio:.4f}")
print()
print("=" * 65)
print("Required UV coupling for each dark gauge group")
print("=" * 65)
print(f"{'Group':>12} {'b0':>8} {'alpha(Mpl)':>12} "
      f"{'1/alpha':>10} {'known?':>15}")
print("-" * 65)

# Known couplings at Mpl for reference
# alpha_gravity ~ 1 (non-perturbative)
# alpha_strong(Mpl) ~ 0.019
# alpha_weak(Mpl)   ~ 0.024
# alpha_EM(Mpl)     ~ 0.007
# alpha_GUT         ~ 0.033

known = {
    'alpha_EM(Mpl)'    : 0.0073,
    'alpha_weak(Mpl)'  : 0.0240,
    'alpha_strong(Mpl)': 0.0191,
    'alpha_GUT'        : 0.0333,
    'alpha=1/2pi'      : 1/(2*np.pi),
    'alpha=1'          : 1.0,
}

interesting = []

for Nc in range(2, 8):
    for Nf in range(0, 2*Nc+1):
        b0 = (11*Nc - 2*Nf)/3.0
        if b0 <= 0:
            continue

        alpha_needed = -2*np.pi / (b0 * log_ratio)

        # Check if matches any known coupling
        match = ""
        for name, val in known.items():
            if abs(alpha_needed/val - 1) < 0.15:
                match = name
                interesting.append(
                    (Nc, Nf, b0,
                     alpha_needed, name))

        if match:
            print(f"  SU({Nc}) Nf={Nf:>2} "
                  f"{b0:>8.3f} "
                  f"{alpha_needed:>12.6f} "
                  f"{1/alpha_needed:>10.3f} "
                  f"{match:>15}")

print()
print("=" * 65)
print("All matches within 15% of known couplings:")
print()

if interesting:
    for Nc, Nf, b0, alpha, name in interesting:
        ratio = alpha / known[name]
        print(f"  SU({Nc}) Nf={Nf:<2}  "
              f"b0={b0:.3f}  "
              f"alpha={alpha:.5f}  "
              f"({ratio:.3f} x {name})")
else:
    print("  No matches found")
    print()
    print("  Scanning what IS required:")
    print()
    print(f"  {'Group':>10} {'alpha_needed':>15} "
          f"{'compared to SM':>20}")
    for Nc in [2,3,4]:
        for Nf in [0,1,2,3]:
            b0 = (11*Nc - 2*Nf)/3.0
            if b0 <= 0:
                continue
            alpha_needed = -2*np.pi/(b0*log_ratio)
            ratio_str = (f"{alpha_needed/0.0191:.2f}"
                         f" x alpha_strong(Mpl)")
            print(f"  SU({Nc}) Nf={Nf}  "
                  f"{alpha_needed:>15.8f}  "
                  f"{ratio_str:>20}")

log(Lambda_dark/Mpl) = -70.6627

Required UV coupling for each dark gauge group
       Group       b0   alpha(Mpl)    1/alpha          known?
-----------------------------------------------------------------
  SU(2) Nf= 3    5.333     0.016672     59.980 alpha_strong(Mpl)
  SU(2) Nf= 4    4.667     0.019054     52.483 alpha_strong(Mpl)
  SU(3) Nf= 0   11.000     0.008083    123.709   alpha_EM(Mpl)
  SU(4) Nf= 1   14.000     0.006351    157.448   alpha_EM(Mpl)
  SU(4) Nf= 2   13.333     0.006669    149.951   alpha_EM(Mpl)
  SU(4) Nf= 3   12.667     0.007020    142.453   alpha_EM(Mpl)
  SU(4) Nf= 4   12.000     0.007410    134.956   alpha_EM(Mpl)
  SU(4) Nf= 5   11.333     0.007846    127.458   alpha_EM(Mpl)
  SU(4) Nf= 6   10.667     0.008336    119.961   alpha_EM(Mpl)
  SU(5) Nf= 7   13.667     0.006506    153.700   alpha_EM(Mpl)
  SU(5) Nf= 8   13.000     0.006840    146.202   alpha_EM(Mpl)
  SU(5) Nf= 9   12.333     0.007210    138.705   alpha_EM(Mpl)
  SU(5) Nf=10   11.667     0.007

In [ ]:
import numpy as np

print("=" * 60)
print("THE TWO EXACT MATCHES")
print("=" * 60)
print()

# Exact matches
matches = [
    ("SU(2)", 2, 4, 4.667, 0.01905, "alpha_strong(Mpl)", 0.0191),
    ("SU(4)", 4, 4, 12.000, 0.00741, "alpha_EM(Mpl)",    0.0073),
    ("SU(5)", 5, 9, 12.333, 0.00721, "alpha_EM(Mpl)",    0.0073),
]

for group, Nc, Nf, b0, alpha_d, sm_name, alpha_sm in matches:
    ratio = alpha_d / alpha_sm
    print(f"  {group} with Nf={Nf} dark flavors")
    print(f"  b0         = {b0:.4f}")
    print(f"  alpha_dark = {alpha_d:.5f}")
    print(f"  alpha_SM   = {alpha_sm:.5f}  ({sm_name})")
    print(f"  ratio      = {ratio:.4f}")
    print()

print("=" * 60)
print("Physical interpretation:")
print()

print("""
  SU(2) Nf=4 match to alpha_strong:
  ----------------------------------
  The dark sector is SU(2) gauge theory
  with 4 dark quarks
  Its UV coupling = QCD coupling at Mpl

  This means SU(2)_dark and SU(3)_QCD
  were UNIFIED at the Planck scale
  They share a common origin

  In Pati-Salam unification:
  SU(4) x SU(2)_L x SU(2)_R
  The SU(2)_dark could be SU(2)_R
  the right-handed weak force
  now hidden and confining

  SU(4) Nf=4 match to alpha_EM:
  ------------------------------
  The dark sector is SU(4) gauge theory
  with 4 dark quarks
  Its UV coupling = EM coupling at Mpl

  In Pati-Salam:
  SU(4) contains both
  color SU(3) and B-L U(1)
  The dark SU(4) could be
  a mirror Pati-Salam color group
""")

print("=" * 60)
print("The Pati-Salam connection:")
print()
print("  Standard Pati-Salam group:")
print("  G_PS = SU(4) x SU(2)_L x SU(2)_R")
print()
print("  Breaking pattern:")
print("  SU(4) → SU(3)_color x U(1)_B-L")
print("  SU(2)_R → U(1)_T3R")
print("  → Standard Model")
print()
print("  DARK Pati-Salam group:")
print("  G_dark = SU(4)' x SU(2)'_L x SU(2)'_R")
print()
print("  If dark sector does NOT break")
print("  SU(4)' remains confined")
print("  SU(2)'_R remains confined")
print("  Their axions = dark energy candidates")
print()

# Now the key prediction
print("=" * 60)
print("KEY PREDICTION:")
print()
print("  If dark energy = axion of")
print("  confined SU(2)_R or SU(4)'")
print("  from a dark Pati-Salam sector")
print()
print("  Then dark matter = dark baryons")
print("  of the same sector")
print()

# Dark baryon mass estimate
# m_baryon ~ Nc * Lambda_dark
Lambda_dark_eV = 2.5e-3
for Nc_dark, group in [(2, "SU(2)"), (4, "SU(4)")]:
    m_baryon_eV = Nc_dark * Lambda_dark_eV * 4*np.pi
    m_baryon_meV = m_baryon_eV * 1e3
    print(f"  {group}' dark baryon mass:")
    print(f"  m ~ Nc * 4pi * Lambda_dark")
    print(f"    ~ {Nc_dark} x 4pi x {Lambda_dark_eV:.2e} eV")
    print(f"    ~ {m_baryon_eV:.4e} eV")
    print(f"    ~ {m_baryon_meV:.4f} meV")
    print()

print("  Ultra-light dark matter")
print("  in the meV range")
print()
print("  Testable with:")
print("  - 21cm cosmology")
print("  - CMB spectral distortions")
print("  - Dark matter direct detection")
print("  at meV energy threshold")

print()
print("=" * 60)
print("UNIFICATION CHAIN:")
print()
print("  Planck scale:")
print("  G_unified = SU(N) or SO(10) or E8")
print("  single coupling alpha_unified")
print()
print("  Breaks to:")
print("  G_SM x G_dark")
print("  SU(3)xSU(2)xU(1) x SU(4)'xSU(2)'")
print()
print("  G_dark confines at 2.5e-3 eV")
print("  producing:")
print("  - TIFA field (dark energy)")
print("  - dark baryons (dark matter)")
print("  - dark photons (dark radiation?)")
print()
print("  ONE sector explains:")
print("  dark energy + dark matter")
print("  from a single confinement scale")
print("  set by dimensional transmutation")
print("  NO fine tuning")
print("  NO cosmological constant problem")

THE TWO EXACT MATCHES

  SU(2) with Nf=4 dark flavors
  b0         = 4.6670
  alpha_dark = 0.01905
  alpha_SM   = 0.01910  (alpha_strong(Mpl))
  ratio      = 0.9974

  SU(4) with Nf=4 dark flavors
  b0         = 12.0000
  alpha_dark = 0.00741
  alpha_SM   = 0.00730  (alpha_EM(Mpl))
  ratio      = 1.0151

  SU(5) with Nf=9 dark flavors
  b0         = 12.3330
  alpha_dark = 0.00721
  alpha_SM   = 0.00730  (alpha_EM(Mpl))
  ratio      = 0.9877

Physical interpretation:


  SU(2) Nf=4 match to alpha_strong:
  ----------------------------------
  The dark sector is SU(2) gauge theory
  with 4 dark quarks
  Its UV coupling = QCD coupling at Mpl
  
  This means SU(2)_dark and SU(3)_QCD
  were UNIFIED at the Planck scale
  They share a common origin
  
  In Pati-Salam unification:
  SU(4) x SU(2)_L x SU(2)_R
  The SU(2)_dark could be SU(2)_R
  the right-handed weak force
  now hidden and confining
  
  SU(4) Nf=4 match to alpha_EM:
  ------------------------------
  The dark sector is SU(4) ga